In [1]:
!pip install openpyxl
!pip install streamlit
!pip install streamlit-option-menu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 13.9 MB/s eta 0:00:00


# Data Merging


In [ ]:
import pandas as pd

# 1. Definisikan path file sesuai lokasi di Google Drive Anda
file_gdp_data = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/GDP_Data_Extract_From_World_Development_Indicators.xlsx'
file_p_data = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/P_Data_Extract_From_World_Development_Indicators.xlsx'
file_democracy = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/democracy-index-eiu.csv'

print("--- TAHAP 1: DATA MERGING ---")
print("Membaca ketiga file...")
# Membaca file Excel (.xlsx)
df_gdp = pd.read_excel(file_gdp_data)
df_p = pd.read_excel(file_p_data)
# Membaca file CSV (.csv) dengan encoding khusus karakter unik
df_dem = pd.read_csv(file_democracy, encoding='latin1')

print("Menggabungkan dataset...")
# Langkah A: Gabungkan file GDP dan P Data (sesama Excel) berdasarkan Country Code
df_merged_gdp = pd.merge(df_gdp, df_p, on='Country Code', how='left', suffixes=('', '_drop'))

# Langkah B: Gabungkan hasilnya dengan file CSV Demokrasi
# Menghubungkan 'Country Code' dari Excel dengan kolom 'Code' dari CSV Demokrasi
df_final_merged = pd.merge(
    df_merged_gdp,
    df_dem,
    left_on='Country Code',
    right_on='Code',
    how='left'
)

# 2. Simpan hasil penggabungan awal (mentah) ke file baru
raw_output_path = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/dataset_merged_raw.csv'
df_final_merged.to_csv(raw_output_path, index=False)

print(f"Tahap Merging Selesai! File mentah disimpan di: {raw_output_path}")
print(f"Ukuran data mentah: {df_final_merged.shape}\n")

--- TAHAP 1: DATA MERGING ---
Membaca ketiga file...
Menggabungkan dataset...
Tahap Merging Selesai! File mentah disimpan di: /content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/dataset_merged_raw.csv
Ukuran data mentah: (8447, 36)



# Data Cleaning

In [ ]:
import numpy as np

print("--- TAHAP 2: DATA CLEANING ---")

# 1. Baca file mentah hasil dari Bagian 1
raw_input_path = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/dataset_merged_raw.csv'
df = pd.read_csv(raw_input_path)

print("Memulai pembersihan data...")

# A. Mengatasi Warning Pandas dengan mengaktifkan opsi silent downcasting
pd.set_option('future.no_silent_downcasting', True)

# B. Tangani missing value tersembunyi '..' khas World Bank menjadi NaN resmi Python
df.replace('..', np.nan, inplace=True)

# C. Deteksi semua kolom tahun (format: 2020 [YR2020]) dan ubah tipe datanya menjadi numerik (float)
kolom_tahun = [col for col in df.columns if '[YR' in col]
for col in kolom_tahun:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# D. Lakukan penyaringan baris (Filtering)
# Hapus baris yang tidak memiliki nilai pada indeks demokrasi
df_cleaned = df.dropna(subset=['Democracy Index'])
# Hapus baris jika semua kolom tahunnya kosong (tidak ada data ekonomi sama sekali)
df_cleaned = df_cleaned.dropna(subset=kolom_tahun, how='all')

# E. Transformasi/Merapikan nama kolom tahun agar hanya berupa angka (Contoh: '2024 [YR2024]' -> '2024')
clean_columns = {col: col.split(' ')[0] for col in kolom_tahun}
df_cleaned = df_cleaned.rename(columns=clean_columns)

# F. Drop/Buang kolom duplikat pasca-merge atau kolom yang tidak dibutuhkan untuk analisis
kolom_dibuang = ['Country Name_drop', 'Series Name_drop', 'Series Code_drop', 'Code', 'Entity']
df_cleaned = df_cleaned.drop(columns=kolom_dibuang, errors='ignore')

# 2. Simpan dataset yang sudah bersih total dan siap dipakai analisis/visualisasi
clean_output_path = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/dataset_cleaned_final.csv'
df_cleaned.to_csv(clean_output_path, index=False)

print(f"Tahap Cleaning Selesai! Data bersih disimpan di: {clean_output_path}")
print(f"Ukuran Data Akhir (Siap Pakai): {df_cleaned.shape}")
print(df_cleaned.head())

--- TAHAP 2: DATA CLEANING ---
Memulai pembersihan data...
Tahap Cleaning Selesai! Data bersih disimpan di: /content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/dataset_cleaned_final.csv
Ukuran Data Akhir (Siap Pakai): (2805, 31)
  Country Name Country Code                       Series Name  \
0  Afghanistan          AFG  GDP per capita growth (annual %)   
1  Afghanistan          AFG  GDP per capita growth (annual %)   
2  Afghanistan          AFG  GDP per capita growth (annual %)   
3  Afghanistan          AFG  GDP per capita growth (annual %)   
4  Afghanistan          AFG  GDP per capita growth (annual %)   

         Series Code  1990  2000      2015      2016     2017      2018  ...  \
0  NY.GDP.PCAP.KD.ZG   NaN   NaN -1.665057 -0.300121 -0.19557 -1.713743  ...   
1  NY.GDP.PCAP.KD.ZG   NaN   NaN -1.665057 -0.300121 -0.19557 -1.713743  ...   
2  NY.GDP.PCAP.KD.ZG   NaN   NaN -1.665057 -0.300121 -0.19557 -1.713743  ...   
3  NY.GDP.PCAP.KD.ZG 

#Data Transformation

In [ ]:
import pandas as pd


print("--- TAHAP 3: DATA TRANSFORMATION---")

# 1. Baca data yang sudah dibersihkan dari Tahap 2
clean_input_path = "/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/dataset_cleaned_final.csv"
df_clean = pd.read_csv(clean_input_path)

# 2. Pisahkan kolom identitas dengan kolom tahun yang akan di-melt
# Daftarkan semua kolom yang BUKAN angka tahun
kolom_identitas = [col for col in df_clean.columns if not col.isdigit()]
# Daftarkan semua kolom tahun yang berupa angka (2015, 2016, dll)
kolom_tahun_angka = [col for col in df_clean.columns if col.isdigit()]

print("Melakukan transformasi dari Wide Format ke Long Format (Melting)...")
# Mengubah struktur tabel agar tahun turun menjadi baris ke bawah
df_dashboard = pd.melt(
    df_clean,
    id_vars=kolom_identitas,          # Kolom yang posisinya tetap (Country Code, Democracy Index, dll)
    value_vars=kolom_tahun_angka,     # Kolom tahun yang ingin dibongkar ke bawah
    var_name='Year_Data',             # Nama kolom baru untuk menampung angka tahun
    value_name='GDP_Growth_Value'     # Nama kolom baru untuk menampung nilai angka GDP
)

# 3. Sinkronisasi Tahun (Opsional tetapi Sangat Direkomendasikan)
# File 'democracy-index-eiu.csv' bawaannya punya kolom 'Year' sendiri yang sifatnya historis per baris.
# Kita pastikan analisisnya tepat dengan mencocokkan Tahun dari data Demokrasi dan data GDP.
# Kita hanya ambil data di mana tahun laporan demokrasi sama dengan tahun data GDP-nya.
df_dashboard['Year_Data'] = df_dashboard['Year_Data'].astype(int)
df_dashboard_final = df_dashboard[df_dashboard['Year'] == df_dashboard['Year_Data']].copy()

# Hapus salah satu kolom tahun yang ganda agar rapi
df_dashboard_final.drop(columns=['Year_Data'], inplace=True)

# 4. Simpan dataset final khusus untuk Dashboard
dashboard_output_path = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/dataset_ready_for_dashboard.csv'
df_dashboard_final.to_csv(dashboard_output_path, index=False)

print(f"Transformasi Selesai! Data SIAP DASHBOARD disimpan di: {dashboard_output_path}")
print(f"Ukuran Data Siap Dashboard: {df_dashboard_final.shape}")
print(df_dashboard_final.head())

--- TAHAP 3: DATA TRANSFORMATION---
Melakukan transformasi dari Wide Format ke Long Format (Melting)...
Transformasi Selesai! Data SIAP DASHBOARD disimpan di: /content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/dataset_ready_for_dashboard.csv
Ukuran Data Siap Dashboard: (1650, 19)
     Country Name Country Code                       Series Name  \
5617  Afghanistan          AFG  GDP per capita growth (annual %)   
5634      Albania          ALB  GDP per capita growth (annual %)   
5651      Algeria          DZA  GDP per capita growth (annual %)   
5668       Angola          AGO  GDP per capita growth (annual %)   
5685    Argentina          ARG  GDP per capita growth (annual %)   

            Series Code     1990.1      2000.1     2016.1     2017.1  \
5617  NY.GDP.PCAP.KD.ZG        NaN         NaN   4.383892   4.975952   
5634  NY.GDP.PCAP.KD.ZG        NaN    0.050018   1.275432   1.986661   
5651  NY.GDP.PCAP.KD.ZG  16.652534    0.339163   6.397

# Serving Machine Learning


1. ETL Script Feature Engineering

In [ ]:
import pandas as pd
import numpy as np

print("--- TAHAP 4: SERVING MACHINE LEARNING (FEATURE ENGINEERING) ---")

# 1. Baca data yang sudah ditransformasi (Long Format) dari tahap sebelumnya
input_path = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/Transformed/dataset_ready_for_dashboard.csv'
df = pd.read_csv(input_path)

# Pastikan data terurut berdasarkan Negara dan Tahun untuk pembuatan fitur historis
df = df.sort_values(by=['Country Code', 'Year']).reset_index(drop=True)

print("Membuat fitur untuk Machine Learning...")

# A. Membuat Lag Feature (Nilai GDP Tahun Sebelumnya)
# Ini sangat penting dalam ML karena pertumbuhan ekonomi tahun lalu sangat memengaruhi tahun ini
df['GDP_Growth_Lag1'] = df.groupby('Country Code')['GDP_Growth_Value'].shift(1)
df['Democracy_Index_Lag1'] = df.groupby('Country Code')['Democracy Index'].shift(1)

# B. Membuat Fitur Kategorikal Menjadi Angka (One-Hot Encoding untuk Region)
# Model ML tidak bisa membaca teks seperti "Asia" atau "Europe", harus diubah ke angka
df_ml = pd.get_dummies(df, columns=['World region according to OWID'], prefix='region', dtype=int)

# C. Menghapus baris NaN yang muncul akibat proses 'shift' (tahun paling awal dari tiap negara)
df_ml = df_ml.dropna(subset=['GDP_Growth_Lag1', 'Democracy_Index_Lag1'])

# D. Menentukan Target Variabel (Variabel yang ingin diprediksi oleh ML)
# Target kita adalah GDP_Growth_Value saat ini.
# Kita pastikan kolom target tidak memiliki nilai kosong (NaN)
df_ml = df_ml.dropna(subset=['GDP_Growth_Value'])

# 2. LOAD: Simpan dataset khusus ini ke target folder ML
ml_output_path = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/features_ready_for_ml.csv'
df_ml.to_csv(ml_output_path, index=False)

print(f"Serving ML Sukses! Fitur siap latih disimpan di: {ml_output_path}")
print(f"Ukuran Matriks Fitur ML: {df_ml.shape}")

--- TAHAP 4: SERVING MACHINE LEARNING (FEATURE ENGINEERING) ---
Membuat fitur untuk Machine Learning...
Serving ML Sukses! Fitur siap latih disimpan di: /content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/features_ready_for_ml.csv
Ukuran Matriks Fitur ML: (1465, 26)


2. Demo Serving ML


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import pandas as pd

print("--- DEMO CONSUMING SERVED ML DATA ---")

# 1. Ambil data dari target serving ETL kita
df_features = pd.read_csv('/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS/features_ready_for_ml.csv')

# 2. Pisahkan Fitur (X) yard Target (y)
# Kita gunakan wilayah (region) hasil encoding, Democracy Index saat ini, serta data historis Lag1
kolom_fitur = [col for col in df_features.columns if col.startswith('region_')] + ['Democracy Index', 'GDP_Growth_Lag1', 'Democracy_Index_Lag1']
X = df_features[kolom_fitur]
y = df_features['GDP_Growth_Value']

# 3. Split Data menjadi Train dan Test Set (Perbaikan pada parameter test_size)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Latih Model Baseline Machine Learning (Linear Regression)
model = LinearRegression()
model.fit(X_train, y_train)

# 5. Evaluasi Singkat
y_pred = model.predict(X_test)
rmse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n[SUCCESS] Model Machine Learning Berhasil Dilatih Menggunakan Hasil Data ETL!")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"R-squared Score (R2): {r2:.4f}")

--- DEMO CONSUMING SERVED ML DATA ---

[SUCCESS] Model Machine Learning Berhasil Dilatih Menggunakan Hasil Data ETL!
Root Mean Squared Error (RMSE): 25.0286
R-squared Score (R2): 0.1307


# PIPELINE ETL MASTER


In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os

# ==========================================
# CONFIGURATION (Mengatur Path Folder UAS)
# ==========================================
BASE_DIR = '/content/drive/MyDrive/Data Engineering S4/Draft 2 (stabilitas politik, ekonomi dunia)/UAS'

PATHS = {
    'gdp_excel': os.path.join(BASE_DIR, 'GDP_Data_Extract_From_World_Development_Indicators.xlsx'),
    'inflation_excel': os.path.join(BASE_DIR, 'P_Data_Extract_From_World_Development_Indicators.xlsx'),
    'democracy_csv': os.path.join(BASE_DIR, 'democracy-index-eiu.csv'),
    'database_target': os.path.join(BASE_DIR, 'stabilitas_ekonomi_politik.db')
}

pd.set_option('future.no_silent_downcasting', True)

# =========================================================================
# FUNGSI PEMBANTU: Memproses & Melakukan Melt File World Bank secara Terpisah
# =========================================================================
def process_world_bank_file(file_path, value_name):
    try:
        df = pd.read_excel(file_path)
        # 1. Bersihkan missing value tersembunyi '..'
        df.replace('..', np.nan, inplace=True)

        # 2. Identifikasi kolom tahun asli (Format WDI: '2020 [YR2020]')
        kolom_tahun = [col for col in df.columns if '[YR' in col]
        for col in kolom_tahun:
            df[col] = pd.to_numeric(df[col], errors='coerce')

        # 3. Hapus baris metadata/kosong di bagian bawah (baris 3000-8000an)
        df = df.dropna(subset=['Country Code'])

        # 4. Melt dari Wide ke Long Format
        df_long = pd.melt(
            df,
            id_vars=['Country Name', 'Country Code'],
            value_vars=kolom_tahun,
            var_name='Year_Raw',
            value_name=value_name
        )

        # 5. Rapikan nama tahun (Contoh: '2020 [YR2020]' -> 2020)
        df_long['Year'] = df_long['Year_Raw'].apply(lambda x: int(x.split(' ')[0]))
        df_long.drop(columns=['Year_Raw'], inplace=True)

        return df_long
    except Exception as e:
        print(f"❌ Gagal memproses file {file_path}: {str(e)}")
        raise

# ==========================================
# CORE ETL PIPELINE
# ==========================================
def run_etl_pipeline():
    print("=== RUNNING AUTOMATED ETL DATA PIPELINE (WITH INFLATION) ===")

    # 1. EXTRACT & TRANSFORM DATA WORLD BANK
    print("\n[1/4] Mengekstrak & Melt Data GDP serta Inflasi secara terpisah...")
    df_gdp_long = process_world_bank_file(PATHS['gdp_excel'], 'GDP_Growth_Value')
    df_inf_long = process_world_bank_file(PATHS['inflation_excel'], 'Inflation_Value')

    # Gabungkan data GDP dan Inflasi yang sudah berformat panjang
    print("Menggabungkan data GDP dan data Inflasi...")
    df_wb_combined = pd.merge(df_gdp_long, df_inf_long, on=['Country Code', 'Year'], how='outer')
    df_wb_combined.drop(columns=['Country Name_y'], errors='ignore', inplace=True)
    df_wb_combined.rename(columns={'Country Name_x': 'Country Name'}, inplace=True)

    # 2. EXTRACT & TRANSFORM DATA DEMOKRASI (CSV)
    print("\n[2/4] Mengekstrak & Membersihkan data Indeks Demokrasi...")
    df_dem = pd.read_csv(PATHS['democracy_csv'], encoding='latin1')
    df_dem.dropna(subset=['Code', 'Year'], inplace=True)
    df_dem['Year'] = df_dem['Year'].astype(int)

    # 3. INTEGRASI AKHIR (MERGE ALL)
    print("\n[3/4] Menggabungkan seluruh data (GDP + Inflasi + Demokrasi)...")
    df_final = pd.merge(
        df_wb_combined,
        df_dem,
        left_on=['Country Code', 'Year'],
        right_on=['Code', 'Year'],
        how='inner'  # Inner merge menjamin keselarasan tahun antar dimensi
    )

    # Hapus kolom hasil duplikasi merge
    df_final.drop(columns=['Code', 'Entity'], errors='ignore', inplace=True)
    df_final.dropna(subset=['Democracy Index'], inplace=True)

    # --- PROSES SERVING MACHINE LEARNING (FEATURE STORE) ---
    print("Menyiapkan fitur lanjutan untuk Machine Learning...")
    df_ml = df_final.sort_values(by=['Country Code', 'Year']).reset_index(drop=True)

    # Membuat Fitur Historis (Lag 1 Tahun Sebelumnya) untuk GDP, Inflasi, dan Demokrasi
    df_ml['GDP_Growth_Lag1'] = df_ml.groupby('Country Code')['GDP_Growth_Value'].shift(1)
    df_ml['Inflation_Lag1'] = df_ml.groupby('Country Code')['Inflation_Value'].shift(1)
    df_ml['Democracy_Index_Lag1'] = df_ml.groupby('Country Code')['Democracy Index'].shift(1)

    # One-Hot Encoding untuk kolom wilayah (Region)
    df_ml = pd.get_dummies(df_ml, columns=['World region according to OWID'], prefix='region', dtype=int)

    # Bersihkan baris NaN yang muncul akibat proses pergeseran fitur historis (Lag)
    df_ml.dropna(subset=['GDP_Growth_Lag1', 'Inflation_Lag1', 'Democracy_Index_Lag1', 'GDP_Growth_Value', 'Inflation_Value'], inplace=True)

    # 4. LOAD PHASE (Menyimpan Hasil Akhir)
    print("\n[4/4] Memulai Tahap LOAD (Menulis ke CSV dan Database SQL)...")
    try:
        # Simpan ke CSV untuk Dashboard dan ML
        df_final.to_csv(os.path.join(BASE_DIR, 'dataset_ready_for_dashboard.csv'), index=False)
        df_ml.to_csv(os.path.join(BASE_DIR, 'features_ready_for_ml.csv'), index=False)

        # Simpan ke target utama: Database SQLite (.db)
        conn = sqlite3.connect(PATHS['database_target'])
        df_final.to_sql('tabel_dashboard_analisis', conn, if_exists='replace', index=False)
        df_ml.to_sql('tabel_features_ml', conn, if_exists='replace', index=False)
        conn.close()

        print("\n🚀 [SELESAI] PIPELINE ETL SUKSES 100%!")
        print(f"Kolom baru yang tersedia di database Anda: {list(df_final.columns)}")
        print(f"Total baris data siap pakai: {df_final.shape[0]} baris.")
    except Exception as e:
        print(f"❌ Gagal pada tahap Load: {str(e)}")
        raise

if __name__ == "__main__":
    run_etl_pipeline()

=== RUNNING AUTOMATED ETL DATA PIPELINE (WITH INFLATION) ===

[1/4] Mengekstrak & Melt Data GDP serta Inflasi secara terpisah...
Menggabungkan data GDP dan data Inflasi...

[2/4] Mengekstrak & Membersihkan data Indeks Demokrasi...

[3/4] Menggabungkan seluruh data (GDP + Inflasi + Demokrasi)...
Menyiapkan fitur lanjutan untuk Machine Learning...

[4/4] Memulai Tahap LOAD (Menulis ke CSV dan Database SQL)...

🚀 [SELESAI] PIPELINE ETL SUKSES 100%!
Kolom baru yang tersedia di database Anda: ['Country Name', 'Country Code', 'GDP_Growth_Value', 'Year', 'Inflation_Value', 'Democracy Index', 'World region according to OWID']
Total baris data siap pakai: 1660 baris.
